In [ ]:
import csv
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import os
import pandas as pd
import seaborn as sns




In [ ]:
def gsa_heat_custom(order_effects,
             xlabels, 
             ylabels, 
             savepath, 
             figname="heatmap_gsa",
             title="Total order effects",
             height=10,
             width=10, 
             cbar_shift=None,
             cbar_width=10,
             fontsize=16):

    """
    Plots a heatmap for a global sensitivity analysis.

    Args:
        - order_effects: first or total order effects matrix 
        - xlabels: parameter names
        - ylabels: output names
        - savepath: where to save the figure
        - height: figure height
        - width: figure width
        - correction: if you want to get rid of very small effects (below 0.01)
        - cbar_shift: shift to apply to the colorbar
        - cbar_width: how wide should your colorbar be (set to 0 to get rid of the colorbar)
        - xlabels_latex: latex parameter names - for paper plotting
        - ylabels_latex: latex output names - for paper plotting
        - fontsize: size of figure font

    """

    order_effects = order_effects / np.sum(order_effects, axis=0)

    plt.rc('text', usetex=False)

    if cbar_shift is None:
        cbar_shift = 0.0

    fig, axes = plt.subplots(figsize=(width, height))
    df = pd.DataFrame(data=np.transpose(order_effects), index=ylabels, columns=xlabels)

    rot_angle_x = 90
    rot_angle_y = 0
    cbar_size = 0.8
    cbar_orientation = "vertical"

    h1 = sns.heatmap(
        df,
        cmap="rocket_r",
        vmin=0.0,
        vmax=1.0,
        square=True,
        linewidth=0.5,
        linecolor='black',
        cbar=False,
        ax=axes,
    )

    if cbar_width > 0:
        divider = make_axes_locatable(axes)
        cax = divider.append_axes("right", size="2.5%", pad=0.1)

        cbar = fig.colorbar(h1.get_children()[0], cax=cax, orientation="vertical", shrink=cbar_size)
        cbar.outline.set_visible(False)
        cbar.set_label("$ST$", fontsize=fontsize, rotation=-90, labelpad=10)
        cbar.set_ticks([0, 1])
        cbar.ax.set_yticklabels(['0', '1'], fontsize=fontsize)
        cbar.ax.tick_params(size=0)  # Remove tick marks

    h1.set_xticks(np.arange(df.shape[1]) + 0.5)
    h1.set_yticks(np.arange(df.shape[0]) + 0.5)
    axes.set_title(title, fontsize=fontsize+5, fontweight="bold", pad=30)  # Increase space below the title
    axes.tick_params(left=False, bottom=False)

    h1.set_xticklabels(xlabels, rotation=rot_angle_x, va="top", fontsize=fontsize)
    h1.set_yticklabels(ylabels, rotation=rot_angle_y, ha="right", fontsize=fontsize)

    plt.tight_layout()
    plt.savefig(f"{savepath}/{figname}.png", bbox_inches="tight", dpi=300)


In [ ]:
heart_names = ["2","3","4","5"]
scenario_numbers = {"EP": ["37","38","39","40"],
                    "inflation": ["41","42","43","44"]}
harddrive = "Elements"
simulation_type = "EP"

for i in range(len(heart_names)):
	case_name = heart_names[i]
	scenario_number = scenario_numbers[simulation_type][i]

	scenario = f"HCM/{case_name}/scenarios/{scenario_number}"

	# Read Si_total.csv and convert to numpy array of floats
	with open(f"/path/to/data/{harddrive}/{scenario}/output/Si_total.csv", 'r') as f:
		csv_reader = csv.reader(f)
		ST_all = np.array([list(map(float, row)) for row in csv_reader])

	if os.path.isfile(f"/path/to/data/{harddrive}/{scenario}/data/xlabels_plot.txt"):
		xlabels = np.loadtxt(f"/path/to/data/{harddrive}/{scenario}/data/xlabels_plot.txt", dtype=str)
	else:
		xlabels = np.loadtxt(f"/path/to/data/{harddrive}/{scenario}/data/xlabels.txt", dtype=str)
	if os.path.isfile(f"/path/to/data/{harddrive}/{scenario}/data/ylabels_plot.txt"):
		ylabels = np.loadtxt(f"/path/to/data/{harddrive}/{scenario}/data/ylabels_plot.txt", dtype=str)
	else:
		ylabels = np.loadtxt(f"/path/to/data/{harddrive}/{scenario}/data/ylabels.txt", dtype=str)

	plotpath=f"/path/to/data/{harddrive}/{scenario}/figures/"

	os.makedirs(plotpath, exist_ok=True)

	gsa_heat_custom(order_effects = ST_all, 
					title=f"Passive mechanics sensitivity analysis for heart #{i+1}",
					xlabels=xlabels, 
					ylabels=ylabels, 
					fontsize=14,
					savepath=plotpath, 
					figname=f"{simulation_type}_heatmap_GSA_{i+1}",
					height=10,
					cbar_width=6)


# Radar charts to compare GSAs

In [ ]:
import csv
import matplotlib.pyplot as plt
import numpy as np



def plot_GSA_radar_chart(scenarios, xlabels, ylabels, savepath, fontsize,figname_preffix, legend = [], colors = [], harddrives=[]):

    for output_idx in range(len(ylabels)):
        figname = f"{figname_preffix}_{output_idx}"

        theta = xlabels
        theta_radians = np.linspace(0, 2 * np.pi, len(theta), endpoint=False).tolist()
        theta_radians += theta_radians[:1]

        r = np.zeros((len(scenarios), len(theta)))


        for i, scenario in enumerate(scenarios):
            # Read Si_total.csv and convert to numpy array of floats
            with open(f"/path/to/data/{harddrives[i]}/HCM/{scenario}/output/Si_total.csv", 'r') as f:
                csv_reader = csv.reader(f)
                ST_all = np.array([list(map(float, row)) for row in csv_reader])

            order_effects = ST_all[:,output_idx]
            order_effects = order_effects / np.sum(order_effects)

            r[i] = order_effects
            
        r = np.concatenate((r, r[:, :1]), axis=1)
        # Define colorblind-safe colors
        # colors = ['#377eb8', '#ff7f00', '#4daf4a',
        #                 '#f781bf', '#a65628', '#984ea3',
        #                 '#999999', '#e41a1c', '#dede00']

        # Curran et al's colours
        # colors = ['#4fcdba', '#e662ad', '#f7c26f', '#6ecb74', '#173263']

        if len(colors) == 0:
            colors = ['#e662ad', '#f7c26f', '#6ecb74', '#173263']

        # Create polar plot
        fig, ax = plt.subplots(subplot_kw={'projection': 'polar'})

        if len(legend) == 0:
            legend = [f"Heart #{i+1}" for i in range(len(scenarios))]

        for i, scenario in enumerate(scenarios):
            ax.plot(theta_radians, r[i], marker='.', label=legend[i], color=colors[i])

# Set theta ticks and labels
        ax.set_xticks(theta_radians[:-1])
        ax.set_xticklabels(theta, fontsize=fontsize)
        ax.xaxis.set_tick_params(which="major", pad=10)

        ax.set_rmax(1)

        title = f"{ylabels[output_idx]}"
        if title.startswith("$"):
            title = f"$\mathbf{{{title[1:-1]}}}$"

        plt.title(title, fontsize=fontsize+5, fontweight="bold", pad=30)
        plt.legend(loc="upper right", bbox_to_anchor=(1.5, 1.2))

        plt.tight_layout()

        os.makedirs(savepath,exist_ok=True)
        plt.savefig(f"{savepath}/{figname}.png", bbox_inches="tight", dpi=300)

In [ ]:
simulation_type = "inflation"  # or "inflation"
heart_names = ["1", "2","3","4","5"]
scenario_numbers = {"EP": ["51","37","38","39","40"],
					"inflation": ["54","41","42","43","44"]}
harddrives = ["../../data","SeagateExpansionDrive","SeagateExpansionDrive","SeagateExpansionDrive","SeagateExpansionDrive"]

# CHOOSE THIS
figname_preffix = f"{simulation_type}_GSA_radarchart"
legend = ["Mid-to-apical LVH", "LVOTO", "Isolated basal LVH", "Milder asymmetric LVH", "Undifferentiated pattern"]
# colors = ["#E57373","#9b2948","#ff7251","#ffcd74","#ffedbf"]
colors = ["#FFD700", "#FFC04D", "#FFA366", "#FF8666", "#FF6666"]





scenarios = [f"{heart_names[i]}/scenarios/{scenario_numbers[simulation_type][i]}" for i in range(len(heart_names))]

xlabels = np.loadtxt(f"/path/to/data/{harddrives[-1]}/HCM/{scenarios[-1]}/data/xlabels_plot.txt", dtype=str)
ylabels = np.loadtxt(f"/path/to/data/{harddrives[-1]}/HCM/{scenarios[-1]}/data/ylabels_plot.txt", dtype=str)
savepath = f"/path/to/data/Bob/HCM/figures"

fontsize=14

plot_GSA_radar_chart(scenarios=scenarios,
					 xlabels=xlabels,
					 ylabels=ylabels,
					 savepath=savepath,
					 fontsize=fontsize,
					 figname_preffix=figname_preffix,
					 legend=legend,
					 colors=colors,
                     harddrives=harddrives)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

xlabels = ["Input 1", "Input 2", "Input 3", "Input 4", "Input 5", "Input 6"]
ylabels = "Output"
savepath = f"/path/to/data/{harddrive}/HCM/figures"

fontsize = 14
figname = "template_GSA_radarchart"

theta = xlabels
theta_radians = np.linspace(0, 2 * np.pi, len(theta), endpoint=False).tolist()
# Close the circle
theta_radians += theta_radians[:1]

r1 = [1, 0.2, 0.08, 0.24, 0.32, 0.12]
r2 = [0.14, 0.07, 0.23, 0.06, 1, 0.14]

r = np.array([[i/sum(r1) for i in r1],
              [i/sum(r2) for i in r2]])
# Close the circle
r = np.concatenate((r, r[:, :1]), axis=1)

# Define colorblind-safe colors
# colors = ['#377eb8', '#ff7f00', '#4daf4a',
#                 '#f781bf', '#a65628', '#984ea3',
#                 '#999999', '#e41a1c', '#dede00']

# Curran et al's colours
colors = ['#4fcdba', '#e662ad']

# Create polar plot
fig, ax = plt.subplots(subplot_kw={'projection': 'polar'})

for i in range(2):
    ax.plot(theta_radians, r[i], marker='.', label=f"Heart #{i+1}", color=colors[i])

# Set theta ticks and labels
ax.set_xticks(theta_radians[:-1])
ax.set_xticklabels(theta, fontsize=fontsize)
ax.xaxis.set_tick_params(which="major", pad=20)

ax.set_rmax(1)

title = f"{ylabels}"
if title.startswith("$"):
    title = f"$\mathbf{{{title[1:-1]}}}$"

plt.title(title, fontsize=fontsize + 5, fontweight="bold", pad=30)
plt.legend(loc="upper right", bbox_to_anchor=(1.5, 1.2))

plt.tight_layout()
plt.savefig(f"{savepath}/{figname}.png", bbox_inches="tight", dpi=300)


# Rankings

In [ ]:
from gpytGPE.utils.design import read_labels
import re

In [ ]:
def gsa_parameters_ranking_S_free_pathfile(loadpath,
                             loadpath_sobol,
                             gsa_mode="STi",
                             mode="max",
                             threshold_cutoff=0.9,
                             output_file=None,
                             features_file=None,
                             important_params_idx_file=None):

    """
    Ranks parameters using sensitivity indies.

    Args:
        - loadpath: datafolder containing data used for GPE training
        - loadpath_sobol: folder containing GPEs and GSA results
        - gsa_mode: output GSA file to read in 
        - mode: how to rank the parameters [max,mean,sum]
        - output_file: output file to save ranking
        - features_file: features file containing which features to consider
    """

    print('Checking folder structure...')
    to_check = [loadpath + "xlabels.txt",
                loadpath + "ylabels.txt",
                loadpath_sobol]
    for f in to_check:
        if not os.path.exists(f):
            raise Exception('Cannot find file '+f)
        else:
            print(f+' found.')

    xlabels = read_labels(loadpath+"xlabels.txt")
    ylabels = read_labels(loadpath+"ylabels.txt")

    if features_file is None:
        features_file = loadpath + "features_idx_list.txt"

    if not os.path.exists(features_file):
        raise Exception('Cannot find '+features_file)

    features = np.loadtxt(features_file, dtype=int)
    if len(features.shape)==0:
        features = [features]
    else:
        features = list(features)

    print('Ranking parameters using features:')
    for idx in features:
        print(ylabels[idx])

    # S = np.zeros((len(xlabels),len(features)),dtype=float)
    # for i,idx in enumerate(features):
    #     S_idx = np.loadtxt(f"{loadpath_sobol}/{gsa_mode}.txt")
    #     S[:,i] = np.mean(S_idx, axis=0)
    
    with open(f"{loadpath_sobol}/{gsa_mode}.csv", 'r') as f:
                csv_reader = csv.reader(f)
                S = np.array([list(map(float, row)) for row in csv_reader])

    S_total = np.zeros((len(xlabels),1),dtype=float)
    print('Ranking parameters according to their '+mode+' effect...')
    if mode=="mean":
        S_total = np.mean(S,axis=1)
    elif mode=="sum":
        S_total = np.sum(S,axis=1)
    elif mode=="max":
        S_total = np.max(S,axis=1)
    else:
        print("mode not recognised: please choose between mean, max and sum")

    ranked = np.argsort(S_total)
    ranked = ranked[::-1]
    ranked_S = S_total[ranked]

    if output_file is None:
        output_file = loadpath_sobol+"Rank_"+gsa_mode+"_"+mode+".txt"

    f = open(output_file, "w")
    for i in range(len(xlabels)):
        f.write(xlabels[ranked[i]]+"\t"+str(ranked_S[i])+"\n")
    f.close()

    print('Normalising ranked sensitivity to compute explained variance...')
    ranked_S_norm = list(np.array(ranked_S)/sum(ranked_S))

    ranked_S_norm_cumulative = []
    for i in range(len(xlabels)):
        ranked_S_norm_cumulative.append(sum(ranked_S_norm[0:i+1]))

    if output_file is None:
        output_file = loadpath_sobol+"Rank_"+gsa_mode+"_"+mode+"_ExpVariance.txt"
    else:
        output_file = output_file[:-4]+"_ExpVariance.txt"

    f = open(output_file, "w")
    for i in range(len(xlabels)):
        f.write(xlabels[ranked[i]]+"\t"+str(ranked_S_norm[i])+"\t"+str(ranked_S_norm_cumulative[i])+"\n")
    f.close()

    if important_params_idx_file is not None:
        idx_cutoff = np.where(np.array(ranked_S_norm_cumulative)>threshold_cutoff)[0][0]
        idx_param = ranked[range(idx_cutoff+1)]
        np.savetxt(important_params_idx_file,idx_param,fmt="%g") 

def plot_rank_GSA_free_th_color(datapath,
                  loadpath,
                  rank_file=None,
                  criterion="STi",
                  mode="max",
                  figname="",
                  normalise=False,
                  acc_var_th=0.9,
                  th=0.0,
                  annotate=False,
                  figsize=(15,5),
                  fontsize=14,
                  xlabels_latex=None,
                  separate_colors=False,
                  color_important=None,
                  color_all = "#ff8000"):

    """
    Plots the parameter ranking for a .

    Args:
        - datapath: folder with data (xlabels.txt, etc...)
        - loadpath: path where you saved your parameter ranking 
        - rank_file: if you want to provide a different parameter ranking file 
                     that is not in the loadpath folder
        - criterion: STi or Si e.g. total or first order effects to use for ranking
        - mode: max or mean to rank the parameters
        - figname: name of output figure 
        - normalise: if you want to normalise so that the values all sum up to 1 
        - th: threshold to determine which parameters are important and which ones aren't
        - annotate: write numbers on top of each bar
        - figsize: size of output figure
        - fontsize: size of figure font
        - xlabels_latex: parameter names in latex for paper plots
        - separate_colors: if you want a different colour for important and unimportant parameters
        - color_important: what colour you want the important parameter bars to be
    """

    color = [color_all]

    # assumes that xlabels and ylabels are the same for all tests
    index_i = read_labels(datapath + "/xlabels.txt")

    x = np.arange(len(index_i))
    barWidth = 0.25

    fig, ax = plt.subplots(1, 1, figsize=figsize, constrained_layout=True)
        
    if criterion == "Si":
        tag = "first-order"
    elif criterion == "STi":
        tag = "total"

    if rank_file is None:
        rank_file = loadpath+"/Rank_"+criterion+"_"+mode+".txt"

    f = open(rank_file,"r")
    lines = f.readlines()

    r_dct = {}
    for line in lines:
        line_split = re.split(r'\t+', line)
        r_dct[line_split[0]] = float(line_split[1])

    bars = []
    for l in index_i:
        bars.append(r_dct[l])
    idx_sorted = np.argsort(np.array(bars))

    r = [xx + barWidth for xx in x]

    bars_sorted = [bars[idx] for idx in idx_sorted]
    bars_sorted = bars_sorted[::-1]
    if xlabels_latex is not None:
        index_i_sorted = [xlabels_latex[idx] for idx in idx_sorted]
    else:
        index_i_sorted = [index_i[idx] for idx in idx_sorted]
    index_i_sorted = index_i_sorted[::-1]

    bars_sorted_norm = list(np.array(bars_sorted)/sum(bars_sorted))

    bars_sorted_sum = []
    for i in range(len(bars_sorted)):
        bars_sorted_sum.append(sum(bars_sorted_norm[0:i+1]))
    print(bars_sorted_sum)

    if normalise:
        barplot = bars_sorted_norm
    else:
        barplot = bars_sorted

    plt.xticks(x+barWidth, index_i_sorted, rotation=90,fontsize=fontsize)
    ax.tick_params(axis='both',labelsize=fontsize)

    cutoff_param = np.where(np.array(bars_sorted_sum)>acc_var_th)[0][0]
    if color_important is None:
        color_important = "darkorange"
    color_unimportant = "lightgray"
    colors = [color_important,]*(cutoff_param+1)+[color_unimportant,]*(len(bars_sorted_sum)-cutoff_param-1)

    if separate_colors:
        bars = ax.bar(r, barplot, width=barWidth, edgecolor='white',color=colors)
    else:
        bars = ax.bar(r, barplot, color=color, width=barWidth, edgecolor='white')

    # if xlabels_latex is not None:
    #     plt.rc('text', usetex=False)
    #     # plt.rc('font', family='serif')
    # else:
    #     plt.rc('text', usetex=False)
    if criterion=='STi':
        ax.set_ylabel('$ST$',fontsize=fontsize)
    else:
        ax.set_ylabel('$S_1$',fontsize=fontsize)
    
    if th > 0:
        ax.plot([-2*barWidth,len(index_i)+2*barWidth],[th,th],color='black',linestyle='--')
    
    plt.legend()

    y_max = np.ceil(np.max(np.array(barplot))*10)/10
    yticks = np.arange(0,y_max+0.05,0.05, dtype=float)



    cutoff = r[cutoff_param]+2*barWidth
    ax.plot([cutoff,cutoff],[0,y_max],color='black',linestyle='--')

    if annotate:
        for i,bar in enumerate(bars):
            yval = bar.get_height()
            ax.text(bar.get_x(), yval + .005, str(round(bars_sorted_sum[i]*100))+'%',fontsize=fontsize)

    # ax.set_yticks(yticks)
    # ax.set_yticklabels(np.round(yticks,2),fontsize=16)

    if figname == "":
        plt.show()
    else:
        plt.savefig(figname)

In [ ]:

heart="5"
scenario="50_more_samples"

base_path=f"/path/to/data/Bob/HCM/{heart}/scenarios/{scenario}"
figures_path = f"{base_path}/figures"

gsa_parameters_ranking_S_free_pathfile(loadpath = f"{base_path}/data/",
                                                loadpath_sobol = f"{base_path}/output/",
                                                gsa_mode                  = "Si_total",
                                                mode                      = "max",
                                                threshold_cutoff          = 0.,
                                                output_file               = None,
                                                features_file             = None,
                                                important_params_idx_file = None)

plot_rank_GSA_free_th_color(datapath        = f"{base_path}/data/",
                           loadpath        = f"{base_path}/output/",
                           rank_file       = None,
                           criterion       = "Si_total",
                           mode            = "max",
                           figname         = os.path.join(figures_path,"Rank_max.png"),
                           normalise       = False,
                           th              = 0,
                           annotate        = True,
                           figsize         = (15,5),
                           fontsize        = 14,
                           xlabels_latex   = None,
                           separate_colors = True,
                           color_important = None,
						   acc_var_th=0.69)
